# Step 02. CARLA 기본 실습 1 — 서버 연결과 World 제어

> **참고**: NGV_E2E_Education (Hanyang Univ.) — author Seokhwan Jeong (shjeong00@hanyang.ac.kr)  
> CARLA 0.9.16  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Client로 서버에 연결해 World 정보를 읽고, 동기 모드·맵·날씨를 바꿔봅니다.

> ⚠️ **이 노트북의 코드 셀은 CARLA 시뮬레이터가 실행 중이어야 동작합니다.**  
> 터미널에서 `./CarlaUE4.sh` 로 서버를 먼저 켠 뒤, 셀을 위에서부터 하나씩 실행하세요.

## 학습 목표

1. `carla.Client`로 서버 연결
2. World에서 정보 읽기 (weather, blueprint, spectator)
3. Spectator 시점 이동
4. 동기 모드 설정
5. 맵·날씨 변경

## 1. 서버에 연결

### 이론 배경

`carla.Client(IP, port)` 로 클라이언트를 만들고 `set_timeout` 으로 응답 대기 시간을 정합니다.  
`get_world()` 로 현재 시뮬레이션(World)을 가져옵니다.

In [ ]:
import carla

# Client 생성 후 서버에 연결 (기본 localhost:2000)
client = carla.Client('localhost', 2000)
client.set_timeout(5.0)            # 응답 대기 최대 5초

world = client.get_world()         # 현재 World 가져오기
print('연결 성공 — 현재 맵:', world.get_map().name)


## 2. World 정보 읽기

### 이론 배경

World는 날씨, Blueprint Library, Spectator, Map 등에 접근하는 진입점입니다.  
`get_blueprint_library().filter('vehicle.*')` 로 차량 설계도만 필터링할 수 있습니다.

In [ ]:
# World에서 핵심 정보 읽어오기
weather           = world.get_weather()
blueprint_library = world.get_blueprint_library()
spectator         = world.get_spectator()

print('날씨 cloudiness:', weather.cloudiness)
print('차량 블루프린트 수:', len(blueprint_library.filter('vehicle.*')))
print('현재 spectator 위치:', spectator.get_transform().location)


## 3. Spectator 시점 이동

### 이론 배경

Spectator는 관전 카메라입니다. `Transform(Location, Rotation)` 으로 위치·자세를 지정합니다.  
`pitch=-90` 은 수직으로 내려다보는 탑뷰입니다.

In [ ]:
# Spectator(관전 카메라) 위치 옮기기 — Location + Rotation
location = carla.Location(x=0.0, y=0.0, z=50.0)        # 50m 상공
rotation = carla.Rotation(pitch=-90.0, yaw=0.0, roll=0.0)  # 수직 내려다보기
spectator.set_transform(carla.Transform(location, rotation))
print('spectator를 (0,0,50)에서 바닥을 내려다보게 이동')


## 4. 동기 모드 vs 비동기 모드

### 이론 배경

- **비동기**: 서버가 실시간으로 알아서 진행 → **노트북 셀 단위 실험에 적합**
- **동기**: `world.tick()` 호출마다 1프레임 → 재현성 높지만 **연속 루프**가 필요

> ⚠️ 동기 모드 + `world.tick()` 루프를 노트북 셀로 쪼개면 freeze/deadlock이 납니다.  
> **연속 주행·시나리오는 `scripts/` 의 .py 스크립트로 실행**하세요 (한 프로세스에서 tick 루프 + 안전한 정리).

In [ ]:
# 노트북에서는 비동기(asynchronous) 모드 그대로가 안전합니다.
# (동기 모드 + world.tick() 루프는 셀로 쪼개면 freeze/deadlock -> scripts/ 의 .py 로 실행)
# 한 프레임이 지나가길 "기다리기"만 할 때는 wait_for_tick() 사용 (비동기 전용):
snapshot = world.wait_for_tick()      # 서버가 한 프레임 진행할 때까지 대기 (블록 X)
print('현재 프레임:', snapshot.frame, '/ 비동기 모드에서는 서버가 알아서 진행')
print('동기 모드가 필요한 실습(연속 주행/시나리오)은 scripts/ 의 .py 로 실행하세요:')
print('  python3 scripts/run_syncmode.py')

## 5. 맵 변경

### 이론 배경

`client.get_available_maps()` 로 목록을 보고 `client.load_world('TownXX')` 로 교체합니다.  
맵 로드는 World를 새로 만들므로 반환값을 다시 받아야 합니다.

In [ ]:
# 맵 변경 — 사용 가능한 맵 목록 확인 후 로드
print('사용 가능한 맵:', [m.split('/')[-1] for m in client.get_available_maps()])

# 예: Town03 으로 변경 (로드에 수 초 걸림)
world = client.load_world('Town03')
print('현재 맵:', world.get_map().name)


## 6. 날씨 변경

### 이론 배경

`WeatherParameters` 의 각 항목(구름·비·노면·태양 각도)을 조절해 `set_weather()` 합니다.  
`carla.WeatherParameters.WetCloudyNoon` 같은 **프리셋**도 있습니다.

다음: Step 03에서 ego 차량을 spawn하고 spectator로 따라가 봅니다.

In [ ]:
# 날씨 변경 — WeatherParameters 의 각 값 조절
weather = world.get_weather()
weather.cloudiness         = 80.0    # 0~100
weather.precipitation      = 60.0    # 0~100 (비)
weather.precipitation_deposits = 50.0  # 0~100 (젖은 노면)
weather.wetness            = 40.0
weather.sun_altitude_angle = 25.0    # -90(밤)~90(정오)
weather.sun_azimuth_angle  = 120.0
world.set_weather(weather)
print('흐리고 비 오는 날씨로 변경 완료')

# 프리셋도 사용 가능:
# world.set_weather(carla.WeatherParameters.WetCloudyNoon)
